# NepError — Notebook 3: Llama 3.1 8B Inference
## Frontier LLM reference via zero-shot prompting on Nepali NLI

---

**Project:** NepError: A Hierarchical Diagnostic Taxonomy of Failure Modes in Nepali NLI  
**Author:** Pema Tshering Sherpa, RV University Bengaluru  
**Collaborator:** Prof. Bal Krishna Bal, KU ILP Lab, Kathmandu University  

---

### Why Llama 3.1 8B?

XLM-R and mDeBERTa are **fine-tuned encoders** — they were explicitly trained on NLI labels.  
CHiPSAL-BERT is fine-tuned on MNLI-Nepali in Notebook 2.  

Llama 3.1 8B is a **generative decoder** tested **zero-shot** — no NLI supervision, no  
Nepali-specific training. It reasons about the label from the prompt alone.  
This is the **frontier LLM ceiling**: if Llama still fails on a case, it is a genuinely hard  
instance that even scale and instruction-tuning cannot rescue.

This contrast is a core contribution of NepError:

| Model type | Training signal | What failures tell you |
|---|---|---|
| XLM-R / mDeBERTa | NLI fine-tuned | Architecture + cross-lingual transfer limits |
| CHiPSAL-BERT | Nepali MLM + NLI fine-tune | Nepali-specific encoder limits |
| Llama 3.1 8B | Zero-shot prompt | Scale + reasoning limits — hardest failures |

### Prompting strategy

We use a **structured zero-shot prompt** in English with the Nepali text embedded.  
The model is asked to output exactly one word: `entailment`, `neutral`, or `contradiction`.  
We parse this output and fall back to `neutral` if the model produces unexpected text.

### Two access methods — choose one

| Method | Speed | Cost | Setup |
|--------|-------|------|-------|
| **Groq API** (recommended) | Very fast (~200 tok/s) | Free tier: 14,400 req/day | Just an API key |
| **HuggingFace local** | Slow on T4 (~5 tok/s) | Free | ~16GB download, needs Colab Pro |

**Recommendation:** Use Groq. It runs Llama 3.1 8B on their LPU hardware for free.  
Get your key at: https://console.groq.com

### What this notebook does
1. Load MNLI-Nepali test split  
2. Run Llama 3.1 8B zero-shot NLI inference (via Groq or local HF)  
3. Collect all wrong predictions with the full prompt+response logged  
4. Save `results/failures_llama_mnli.csv`  
5. Compare Llama failure patterns against XLM-R and mDeBERTa failures  

> **Note on subset size:** Running all 10,200 MNLI test rows against an LLM API is  
> time-consuming and hits rate limits. We run on a **stratified 1,000-row subset** by default  
> (333 per label). Change `EVAL_SUBSET_SIZE` to `None` to run the full test set.

---
## Section 0 — Setup & API Configuration

In [ ]:
!pip install -q groq datasets pandas numpy scikit-learn tqdm

In [ ]:
import os, re, time, json, warnings, random
from pathlib import Path
from typing import Optional, Tuple

import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.metrics import classification_report
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

Path('results').mkdir(exist_ok=True)
print("Imports ready.")

In [ ]:
# ─────────────────────────────────────────────────────────
#  CONFIGURATION — set your choices here
# ─────────────────────────────────────────────────────────

# Access method: 'groq' (recommended) or 'local' (needs Colab Pro + 16GB VRAM)
ACCESS_METHOD = 'groq'

# ── Groq settings ──────────────────────────────────────────
# Get your free API key at https://console.groq.com
# Paste it here or set as an environment variable GROQ_API_KEY
GROQ_API_KEY  = os.environ.get('GROQ_API_KEY', 'YOUR_GROQ_API_KEY_HERE')
GROQ_MODEL    = 'llama-3.1-8b-instant'   # Llama 3.1 8B on Groq LPU

# Rate limiting — Groq free tier: 30 requests/minute, 14,400 requests/day
GROQ_REQUESTS_PER_MINUTE = 28   # stay under the 30 rpm limit with margin
GROQ_RETRY_WAIT_SECONDS  = 65   # wait time when rate limit is hit

# ── Local HuggingFace settings (only used if ACCESS_METHOD = 'local') ──
LOCAL_MODEL_ID = 'meta-llama/Meta-Llama-3.1-8B-Instruct'
# Note: requires a HuggingFace account and access approval at
# https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct
HF_TOKEN = os.environ.get('HF_TOKEN', 'YOUR_HF_TOKEN_HERE')

# ── Evaluation settings ────────────────────────────────────
# Number of test instances to evaluate. Set to None for full test set (10,200).
# 1,000 rows takes ~35 minutes on Groq free tier respecting rate limits.
EVAL_SUBSET_SIZE = 1000

# Generation settings
MAX_NEW_TOKENS = 10    # we only need one word back
TEMPERATURE    = 0.0   # greedy decoding for reproducibility

LABEL_NAMES = {0: 'entailment', 1: 'neutral', 2: 'contradiction'}
print("Config loaded.")
print(f"Access method : {ACCESS_METHOD}")
print(f"Subset size   : {EVAL_SUBSET_SIZE if EVAL_SUBSET_SIZE else 'Full test set (10,200 rows)'}")

---
## Section 1 — Load Dataset

In [ ]:
print("Loading IRIIS-RESEARCH/MNLI-Nepali ...")
raw  = load_dataset('IRIIS-RESEARCH/MNLI-Nepali')
test_full = raw['test'].to_pandas()

# Add stable instance IDs for cross-notebook tracking
test_full.insert(0, 'instance_id',
    [f"MNLI-Nepali_test_{i}" for i in range(len(test_full))])

if EVAL_SUBSET_SIZE is not None:
    # Stratified sample: equal rows per label
    n_per_class = EVAL_SUBSET_SIZE // 3
    test_df = (
        test_full
        .groupby('label', group_keys=False)
        .apply(lambda g: g.sample(n=n_per_class, random_state=SEED))
        .reset_index(drop=True)
    )
    print(f"Using stratified subset: {len(test_df):,} rows ({n_per_class} per label)")
else:
    test_df = test_full.copy()
    print(f"Using full test set: {len(test_df):,} rows")

print(f"Label distribution: {test_df['label'].value_counts().sort_index().to_dict()}")

---
## Section 2 — Prompt Engineering

In [ ]:
# ── Prompt Design ─────────────────────────────────────────
#
# We use a zero-shot prompt with explicit label definitions and
# a strict output instruction. The prompt is in English because
# Llama 3.1 8B is instruction-tuned primarily on English data —
# using English instructions with Nepali text maximises the model's
# ability to follow the format while still testing its Nepali
# language understanding.
#
# Label definitions are written out explicitly to avoid the model
# relying on NLI jargon it may not associate with Nepali text.

SYSTEM_PROMPT = """You are an expert in natural language understanding.
Your task is to determine the logical relationship between two Nepali sentences.
You must respond with exactly ONE word — nothing else."""

def build_user_prompt(premise: str, hypothesis: str) -> str:
    return f"""Given the following two sentences in Nepali:

Premise: {premise}
Hypothesis: {hypothesis}

Determine the relationship:
- If the hypothesis LOGICALLY FOLLOWS from the premise, respond: entailment
- If the hypothesis CONTRADICTS the premise, respond: contradiction  
- If the hypothesis is NEITHER guaranteed NOR contradicted by the premise, respond: neutral

Respond with exactly one word (entailment / neutral / contradiction):"""


def parse_label(response_text: str) -> Tuple[int, str]:
    """
    Parse the model's text response into a label integer.

    Returns (label_int, raw_text).
    Falls back to label 1 (neutral) if response is unrecognised —
    neutral is the safest default (neither confirms nor denies).
    """
    text = response_text.strip().lower()
    # Strip punctuation that sometimes sneaks in
    text = re.sub(r'[^a-z]', '', text)

    if text == 'entailment':
        return 0, response_text.strip()
    elif text == 'contradiction':
        return 2, response_text.strip()
    elif text == 'neutral':
        return 1, response_text.strip()
    else:
        # Log unexpected outputs — these are themselves interesting data points
        return 1, f'[PARSE_FAIL: "{response_text.strip()}"]'


# Test the prompt on one example
example = test_df.iloc[0]
print("Example prompt:")
print("─" * 50)
print(f"SYSTEM: {SYSTEM_PROMPT}")
print(f"USER  : {build_user_prompt(example['premise'], example['hypothesis'])}")
print("─" * 50)
print(f"True label: {LABEL_NAMES[example['label']]}")

---
## Section 3 — Inference Client Setup

In [ ]:
if ACCESS_METHOD == 'groq':
    from groq import Groq

    if GROQ_API_KEY == 'YOUR_GROQ_API_KEY_HERE':
        raise ValueError(
            "Please set your Groq API key.\n"
            "Either: GROQ_API_KEY = 'gsk_...' in the config cell above\n"
            "Or set the environment variable: os.environ['GROQ_API_KEY'] = 'gsk_...'"
        )

    groq_client = Groq(api_key=GROQ_API_KEY)
    print("Groq client initialised.")
    print(f"Model: {GROQ_MODEL}")

    def call_llm(premise: str, hypothesis: str) -> Tuple[int, str]:
        """
        Call Llama 3.1 8B via Groq API.
        Retries once on rate limit errors with a 65-second wait.
        """
        for attempt in range(2):
            try:
                response = groq_client.chat.completions.create(
                    model=GROQ_MODEL,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": build_user_prompt(premise, hypothesis)},
                    ],
                    max_tokens=MAX_NEW_TOKENS,
                    temperature=TEMPERATURE,
                )
                raw_text = response.choices[0].message.content
                return parse_label(raw_text)
            except Exception as e:
                if 'rate_limit' in str(e).lower() and attempt == 0:
                    print(f"\n  Rate limit hit. Waiting {GROQ_RETRY_WAIT_SECONDS}s...")
                    time.sleep(GROQ_RETRY_WAIT_SECONDS)
                else:
                    print(f"\n  API error: {e}")
                    return 1, f'[API_ERROR: {str(e)[:50]}]'

elif ACCESS_METHOD == 'local':
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

    if HF_TOKEN == 'YOUR_HF_TOKEN_HERE':
        raise ValueError("Please set your HuggingFace token in the config cell.")

    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Loading {LOCAL_MODEL_ID} locally on {DEVICE} ...")
    print("This will download ~16GB and take several minutes.")

    local_tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_ID, token=HF_TOKEN)
    local_model = AutoModelForCausalLM.from_pretrained(
        LOCAL_MODEL_ID,
        token=HF_TOKEN,
        torch_dtype=torch.float16,
        device_map='auto',
    )
    local_pipeline = pipeline(
        'text-generation',
        model=local_model,
        tokenizer=local_tokenizer,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,       # greedy decoding
        temperature=None,      # required when do_sample=False
        return_full_text=False,
    )
    print("Local model loaded.")

    def call_llm(premise: str, hypothesis: str) -> Tuple[int, str]:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": build_user_prompt(premise, hypothesis)},
        ]
        prompt = local_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        output = local_pipeline(prompt)[0]['generated_text']
        return parse_label(output)

else:
    raise ValueError(f"Unknown ACCESS_METHOD: {ACCESS_METHOD}. Choose 'groq' or 'local'.")

---
## Section 4 — Run Inference

In [ ]:
# ── Rate limiting tracker for Groq ────────────────────────
request_times = []

def rate_limited_call(premise: str, hypothesis: str) -> Tuple[int, str]:
    """Wraps call_llm with request-rate tracking for Groq free tier."""
    if ACCESS_METHOD == 'groq':
        now = time.time()
        # Remove timestamps older than 60 seconds
        request_times[:] = [t for t in request_times if now - t < 60]
        if len(request_times) >= GROQ_REQUESTS_PER_MINUTE:
            wait = 60 - (now - request_times[0]) + 1
            print(f"\n  Rate limit pause: {wait:.0f}s")
            time.sleep(wait)
        request_times.append(time.time())
    return call_llm(premise, hypothesis)


# ── Main inference loop ───────────────────────────────────
results_rows = []
parse_failures = 0

print(f"Running Llama 3.1 8B inference on {len(test_df):,} rows...")
print(f"Estimated time (Groq free tier): ~{len(test_df) * 2.2 / 60:.0f} minutes\n")

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Llama inference"):
    pred_label, raw_response = rate_limited_call(
        row['premise'], row['hypothesis']
    )

    is_parse_fail = raw_response.startswith('[PARSE_FAIL') or \
                    raw_response.startswith('[API_ERROR')
    if is_parse_fail:
        parse_failures += 1

    true_label = int(row['label'])
    results_rows.append({
        'instance_id':    row['instance_id'],
        'premise':        row['premise'],
        'hypothesis':     row['hypothesis'],
        'true_label':     true_label,
        'pred_label':     pred_label,
        'correct':        int(true_label == pred_label),
        'raw_response':   raw_response,   # full model output — useful for error analysis
        'is_parse_fail':  is_parse_fail,
        'model':          'llama-3.1-8b',
        'dataset':        'mnli',
    })

results_df = pd.DataFrame(results_rows)

# Quick diagnostics
accuracy   = results_df['correct'].mean()
n_failures = (results_df['correct'] == 0).sum()

print(f"\n{'='*50}")
print(f"LLAMA 3.1 8B — MNLI-Nepali ({len(results_df):,} rows)")
print(f"{'='*50}")
print(f"Accuracy      : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Failures      : {n_failures:,}")
print(f"Parse failures: {parse_failures}")
print(f"\nClassification Report:")

unique_labels = sorted(results_df['true_label'].unique())
target_names  = [LABEL_NAMES[l] for l in unique_labels]
print(classification_report(
    results_df['true_label'], results_df['pred_label'],
    labels=unique_labels, target_names=target_names, zero_division=0
))

---
## Section 5 — Extract Failure Cases & Save

In [ ]:
import re as _re

NEGATION_MARKERS = [
    'छैन', 'छैनन्', 'दैन', 'दैनन्', 'होइन', 'होइनन्',
    'नाई', 'नगर्', 'नहुने', 'नभएको', 'नगरेको'
]

def is_predominantly_latin(text: str, threshold: float = 0.5) -> bool:
    letters = _re.findall(r'[a-zA-Z\u0900-\u097F]', text)
    if not letters:
        return False
    return (sum(1 for c in letters if c.isascii()) / len(letters)) > threshold

def has_code_switching(text: str) -> bool:
    return bool(_re.search(r'[\u0900-\u097F]', text)) and \
           bool(_re.search(r'[a-zA-Z]{2,}', text))

def has_negation(text: str) -> bool:
    return any(m in text for m in NEGATION_MARKERS)


failures = results_df[results_df['correct'] == 0].copy()

# Stratified sample
n_per_class = min(200, failures['true_label'].value_counts().min())
failures_sampled = (
    failures
    .groupby('true_label', group_keys=False)
    .apply(lambda g: g.sample(n=n_per_class, random_state=SEED))
    .reset_index(drop=True)
)

combined = failures_sampled['premise'] + ' ' + failures_sampled['hypothesis']
failures_sampled['flag_romanization']  = combined.apply(is_predominantly_latin)
failures_sampled['flag_codeswitching'] = combined.apply(has_code_switching)
failures_sampled['flag_negation']      = combined.apply(has_negation)

# Blank annotation columns
failures_sampled['taxonomy_tier']  = ''
failures_sampled['annotator_note'] = ''
failures_sampled['annotator_id']   = ''

out_path = 'results/failures_llama_mnli.csv'
failures_sampled.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f"Total failures     : {len(failures):,}")
print(f"Sampled            : {len(failures_sampled):,}")
print(f"Saved → {out_path}")
print(f"\nPre-filter flags:")
print(f"  flag_negation      : {failures_sampled['flag_negation'].sum()}")
print(f"  flag_codeswitching : {failures_sampled['flag_codeswitching'].sum()}")
print(f"  flag_romanization  : {failures_sampled['flag_romanization'].sum()}")

---
## Section 6 — Cross-Model Failure Comparison

In [ ]:
# ── Load results from Notebooks 1 & 2 to compare ─────────
# Run this cell only after you have all three failure files.

FAILURES_FILES = {
    'XLM-R Large':    'results/failures_xlmr_mnli.csv',
    'mDeBERTa-v3':    'results/failures_mdeberta_mnli.csv',
    'CHiPSAL-BERT':   'results/failures_chipsalbert_mnli.csv',  # from Notebook 2
    'Llama 3.1 8B':   'results/failures_llama_mnli.csv',
}

LABEL_MAP = {0: 'ENT', 1: 'NEU', 2: 'CON'}

comparison_rows = []
for model_name, fpath in FAILURES_FILES.items():
    if not Path(fpath).exists():
        print(f"Skipping {model_name} — file not found: {fpath}")
        continue
    df = pd.read_csv(fpath)
    comparison_rows.append({
        'Model':                model_name,
        'Failures (sampled)':   len(df),
        'Negation-flagged':     df['flag_negation'].sum(),
        'Code-switch-flagged':  df['flag_codeswitching'].sum(),
        'Roman-flagged':        df['flag_romanization'].sum(),
        'Top error':            (
            df.groupby(['true_label','pred_label']).size()
            .reset_index(name='n')
            .sort_values('n', ascending=False)
            .apply(lambda r: f"{LABEL_MAP[r['true_label']]}→{LABEL_MAP[r['pred_label']]}", axis=1)
            .iloc[0]
        ),
    })

if comparison_rows:
    comp_df = pd.DataFrame(comparison_rows)
    print("\nCROSS-MODEL FAILURE COMPARISON")
    print("=" * 65)
    print(comp_df.to_string(index=False))
    comp_df.to_csv('results/cross_model_comparison.csv', index=False)
    print("\nSaved → results/cross_model_comparison.csv")
else:
    print("No comparison files found yet. Run Notebooks 1 and 2 first.")

---
## Section 7 — Qualitative Analysis: What Does Llama Get Wrong?

In [ ]:
# Inspect Llama's most confident wrong predictions
# These are the most interesting for the paper: cases where the model
# is decisively wrong, not just uncertain.

print("LLAMA FAILURE ANALYSIS")
print("=" * 60)

# Error transition breakdown
print("\nError transitions (true → predicted):")
failures_full = results_df[results_df['correct'] == 0].copy()
trans = (
    failures_full
    .groupby(['true_label', 'pred_label']).size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)
trans['transition'] = trans.apply(
    lambda r: f"{LABEL_NAMES[r['true_label']]:>15} → {LABEL_NAMES[r['pred_label']]}", axis=1
)
print(trans[['transition', 'count']].to_string(index=False))

# Parse failure inspection
parse_fails = results_df[results_df['raw_response'].str.startswith('[PARSE', na=False)]
if len(parse_fails) > 0:
    print(f"\nParse failures ({len(parse_fails)} cases) — model gave unexpected output:")
    for _, row in parse_fails.head(5).iterrows():
        print(f"  Raw response: {row['raw_response']}")

# Negation failures — key T2a evidence
neg_failures = failures_full[failures_full['flag_negation'] == True]
print(f"\nNegation-flagged failures: {len(neg_failures)}")
print("Sample negation failures (T2a candidates):")
for _, row in neg_failures.head(3).iterrows():
    print(f"  P: {row['premise'][:90]}")
    print(f"  H: {row['hypothesis'][:90]}")
    print(f"  True: {LABEL_NAMES[row['true_label']]}  |  Llama said: '{row['raw_response']}'")
    print()

---
## Next Steps

You now have failure CSVs from all four models:
```
results/failures_xlmr_mnli.csv          ← Notebook 1
results/failures_mdeberta_mnli.csv      ← Notebook 1  
results/failures_chipsalbert_mnli.csv   ← Notebook 2
results/failures_llama_mnli.csv         ← This notebook
results/cross_model_comparison.csv      ← Summary table
```

**Your annotation corpus is complete. Proceed to:**
1. Open each CSV in Google Sheets
2. Apply the 4-tier taxonomy to each failure case (fill `taxonomy_tier` column)
3. Share with your KU co-annotator — they annotate the same 50 pilot cases independently
4. Compute Cohen's Kappa — target κ ≥ 0.70 before proceeding to full annotation